# Projeto 01 — Agente Inteligente de Otimização de Rotas de Distribuição Alimentar Local
## Unidade Curricular de Inteligência Artificial — IPCA (2026/2027)
### Resolução de Problemas de Pickup-and-Delivery com Janelas Temporais e Capacidade (PDPTW)

---

### Grupo 04 — `IA26_P01_G04`
* **Gonçalo Correia** — Nº 22840
* **Bruno Dantas** — Nº 27958
* **Ricardo Ferreira** — Nº 25426

**Repositório Oficial:** [https://github.com/BrunoDantas1/IA26_P01_G04.git](https://github.com/BrunoDantas1/IA26_P01_G04.git)

---

## 1. Introduction

### 1.1. Contexto do Problema e Enquadramento
A transição para cadeias de abastecimento alimentar curtas e sustentáveis coloca desafios logísticos complexos. Pequenos produtores agrícolas locais operam com margens temporais reduzidas de frescura dos produtos, enquanto os consumidores finais e mercados municipais exigem entregas em horários de conveniência bem definidos.

Neste projeto desenvolvemos um **Agente Inteligente de Planeamento de Rotas** para distribuição alimentar local na região do distrito de **Braga** e concelho da **Póvoa de Varzim**. O problema enquadra-se formalmente na classe dos **Pickup-and-Delivery Problems with Time Windows and Capacity Constraints (PDPTW)**, que combinam restrições espaciais, de precedência, temporais e de capacidade.

### 1.2. Atores e Componentes do Sistema
1. **Produtores Locais (*Producers*)**: Locais de origem onde os produtos agrícolas são colhidos/embalados e disponibilizados dentro da janela regulamentar $[\text{availability\_start}, \text{availability\_end}]$.
2. **Centros de Recolha / Armazéns (*CollectionCenters*)**: Plataformas logísticas centrais onde os bens podem ser agregados e que servem de depósito base permanente da frota de veículos.
3. **Locais de Entrega (*DeliveryLocations / Customers*)**: Clientes residenciais ou mercados municipais com janelas de entrega $[\text{delivery\_start}, \text{delivery\_end}]$.
4. **Frota de Veículos (*Vehicles*)**: Carrinhas comerciais com capacidades máximas específicas (1444.7 kg e 1009.8 kg) que realizam as viagens.
5. **Matriz de Distâncias e Tempos (*DistanceMatrix*)**: Baseada na rede de paragens unificadas de `locations.csv`, fornecendo distâncias em km (distância ortodrómica corrigida com fator rodoviário de 1.3) e tempos de viagem em minutos calculados a uma velocidade de 45 km/h.

### 1.3. Objetivos do Projeto e Divisão de Tarefas
* **Secção 1 — Introdução**: Contextualização, identificação e definição do sistema logístico (preenchida).
* **Secção 2 — Exploratory Data Analysis (EDA)**: Carregamento do dataset, validação de integridade referencial e análise da procura face à frota *(a preencher)*.
* **Secção 3 — Modelação e Testes de Viabilidade**: Definição das classes `Request`, `Vehicle`, `State`, `Problem`, função de custo e validador estrito de rotas com testes manuais (*hand-written routes*) *(a preencher)*.
* **Secções 4 e 5 — Problem Solving & Conclusão**: Algoritmos de pesquisa informada (Greedy, A*), pesquisa local (Hill-Climbing) e CSP com coordenação de frota *(a desenvolver na Entrega Final)*.


## 2. Exploratory Data Analysis (EDA)

A análise exploratória de dados foi conduzida sobre o dataset `data` (11 tabelas CSV), coberta em detalhe no notebook [`01_eda_tiny.ipynb`](01_eda_tiny.ipynb) / [`EDA.ipynb`](EDA.ipynb).

---

### 2.1 Visão Geral do Dataset

O dataset contém **11 entidades** que modelam um sistema logístico de distribuição alimentar local:

| Tabela | Registos | Descrição |
|--------|----------|-----------|
| `localities` | 49 | Freguesias e concelhos (Braga / Póvoa de Varzim) |
| `producers` | 5 | Produtores locais (origem dos pedidos de recolha) |
| `products` | 23 | Catálogo de produtos com peso unitário (kg) |
| `collection_centers` | 1 | Único armazém / base da frota de veículos |
| `vehicles` | 2 | Frota (1 refrigerada + 1 standard) |
| `customers` | 8 | Clientes que efetuam encomendas |
| `delivery_locations` | 6 | Locais de entrega (mercados, residências) |
| `orders` | 10 | Cabeçalhos de encomendas |
| `order_items` | 23 | Linhas de encomenda — pedidos individuais (Requests) |
| `locations` | 12 | Catálogo unificado de paragens com lat/lon |
| `distance_matrix` | 132 | Distâncias (km) e tempos (min) entre todos os pares de paragens |

---

### 2.2 Integridade Referencial

A validação exaustiva das chaves primárias e estrangeiras revelou **100% de integridade** no dataset:

- **Chaves Primárias (PKs):** Todas as 10 tabelas com PK apresentam valores únicos e sem nulos (0 duplicados, 0 nulos em todos os campos-chave).
- **Chaves Estrangeiras (FKs):** As 12 relações inter-tabela verificadas não contêm registos órfãos (violações = 0 em todos os casos), nomeadamente:
  - `producers.locality_id` → `localities`
  - `products.producer_id` → `producers`
  - `vehicles.center_id` → `collection_centers`
  - `orders.customer_id` → `customers`, `orders.delivery_location_id` → `delivery_locations`
  - `order_items.order_id` → `orders`, `order_items.product_id` → `products`
  - `distance_matrix.(origin|destination)_location_id` → `locations`

> O dataset está **estruturalmente válido** e pronto para ser carregado pelo modelo sem necessidade de limpeza ou imputação de dados.

---

### 2.3 Análise de Capacidade: Procura vs. Frota

| Métrica | Valor |
|---------|-------|
| Peso total das 23 encomendas | **~131 kg** |
| Capacidade máx. Veículo 1 (ZG-06-DP) | 1 444,70 kg |
| Capacidade máx. Veículo 2 (TY-74-YA) | 1 000,00 kg |
| Capacidade do Centro de Recolha | 5 000,00 kg |

**Conclusão:** A procura total (~131 kg) representa apenas **~9% da capacidade** do veículo menor. A frota atual é amplamente suficiente para servir todo o dataset de uma só vez; o desafio do problema é, portanto, dominado pelas **restrições de janelas temporais** e não pela capacidade de carga.

---

### 2.4 Consistência das Janelas Temporais

A análise cronológica dos `order_items` revelou **anomalias temporais sistémáticas** no dataset sintético:

- **Anomalia detetada:** Em múltiplos artigos, `order_datetime > availability_start`, ou seja, a encomenda foi registada *após* o início da janela de disponibilidade do produtor — o que é fisicamente impossível num cenário real (um cliente não pode encomendar antes de o produtor anunciar disponibilidade).
- **Impacto para o agente:** O algoritmo deve usar `ready_time = max(order_datetime, availability_start)` como instante de início efetivo da janela de recolha.
- As janelas de entrega (`delivery_start`, `delivery_end`) são consistentes e cobrem períodos de 1 a 3 horas no mesmo dia de encomenda.

---

### 2.5 Mapeamento Espacial das Paragens

O mapa de dispersão lat/lon das 12 paragens unificadas (`locations.csv`) mostra:

- **1 Centro de Recolha** (armazém base, Braga) — ponto central da rede.
- **5 Produtores** — distribuídos pela região de Braga e arredores.
- **6 Locais de Entrega** — dispersos por vários concelhos, com distâncias entre 15 km e 90 km do armazém.
- A **matriz de distâncias** (132 entradas = 12×12 pares) usa distância ortodrómica com fator 1.3 e velocidade média de 45 km/h para calcular durações.

---

### 2.6 Síntese para o Planeamento

Os resultados da EDA estabelecem as seguintes conclusões críticas para a Fase 2 (algoritmos):

1. **Dataset íntegro** — sem necessidade de pré-processamento ou correção de dados.
2. **Restrição dominante = janelas temporais**, não capacidade — os algoritmos devem priorizar a viabilidade temporal acima da otimização de carga.
3. **`ready_time` é o instante de início correto** da janela de recolha (`max(order_datetime, availability_start)`).
4. **Grafo de 12 nós** com matriz de distâncias completa — espaço de estados bem definido para os algoritmos de procura (BFS / DFS / A* / Greedy Best-First).


## 3. Modeling & Feasibility Tests

### 3.1. Formulação do Problema
*(A preencher: Classes Request, Vehicle, State e Problem)*

### 3.2. Função de Custo (cost_function)
*(A preencher: Cálculo de distâncias, tempos de viagem e tempos de espera)*

### 3.3. Validador de Viabilidade (route_feasibility_checker)
*(A preencher: Verificação de depósito, precedência, capacidades e janelas temporais)*

### 3.4. Testes com Rotas Escritas à Mão (Hand-Written Routes)
*(A preencher: Demonstração de pelo menos 2 rotas válidas e 2 inválidas)*


## 4. Problem Solving (Algoritmos de IA)

*(A desenvolver na Entrega Final — Fase 2: Greedy Best-First Search, A* Search com 2 heurísticas admissíveis, Hill-Climbing e CSP Backtracking)*


## 5. Result Analysis & Conclusion

*(A desenvolver na Entrega Final — Fase 2: Tabela comparativa de métricas, análise crítica, limitações e conclusões)*
